# Firm Characteristics

Investors often describe a strategy as buying cheap stocks, profitable companies or firms investing conservatively. Before these descriptions can guide an investment, we need to define the measurements and when their inputs were available.

In this notebook, I first explain firm characteristics and investment signals, then calculate the Fama–French characteristics from the prepared accounting and market data. The example follows the inputs through to their measurements, coverage and distributions. [02](02_portfolio_formation.ipynb) then uses these comparisons to form portfolios.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from factor_asset_pricing.characteristics import fama_french_characteristics
NAVY, BLUE, TEAL, AMBER, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':11,
    'axes.titlesize':12, 'axes.labelsize':11, 'xtick.labelsize':10, 'ytick.labelsize':10,
    'legend.fontsize':10, 'figure.dpi':110, 'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False, 'lines.linewidth':1.8,
    'axes.prop_cycle':plt.cycler(color=[BLUE,TEAL,AMBER,NAVY]), 'grid.alpha':.15})

from factor_asset_pricing.wrds_data import load_prepared_data
RAW_WRDS = ROOT / "data/raw/wrds"
PROCESSED = ROOT / "data/processed"


## 1. Firm Characteristics and Investment Signals

### 1.1 What characteristics measure

A firm characteristic measures an aspect of a company or its security at a particular date. It can describe size, valuation, operating performance or trading history. Here $x_{i,t}$ denotes the measurement for firm or security $i$ at date $t$.

The definition determines what we compare. Profitability scaled by sales differs from profitability scaled by equity, while a volatility estimate depends on its return window. We therefore need to state the inputs, units, denominator and observation period.

### 1.2 From measurements to investment signals

A characteristic becomes an investment signal when we specify how it guides positions. Ranking firms by valuation gives a relative comparison; choosing to buy the high- or low-ranked group is a separate investment decision.

A characteristic also differs from a factor exposure. The characteristic describes a firm, while a beta describes how its returns co-move with a benchmark. A characteristic-based strategy can produce a traded factor, whose exposure and abnormal return we subsequently evaluate.

### 1.3 Information availability and look-ahead bias

A historical strategy can only use information available before choosing its holdings:

$$x_{i,t-1}\in\mathcal F_{t-1}.$$

The fiscal date identifies the period covered by the accounts, the publication date when they became accessible, and the formation date when we use them. Look-ahead bias arises when a historical decision uses information known only afterward. December year-end accounts cannot automatically guide investment in that December.

A formation lag separates measurement from investment, but does not establish actual publication dates or remove later revisions. The example uses a stated availability convention because historical publication dates are unavailable.

### 1.4 Measurement choices and comparability

Comparable measurements require consistent units, dates and identifiers. Market values in thousands must be converted before dividing accounting values in millions. Missing inputs can prevent a calculation, while treating a missing expense as zero changes the numerator. Small denominators can also produce extreme ratios.

Annual growth requires consecutive fiscal years for the same company. Company accounts and security prices describe different entities when a firm has several securities, so we retain separate company and security identifiers. The coverage and distribution summaries show which observations these measurement rules allow us to compare.

## 2. Example: Fama–French Characteristics

I use size, book-to-market, operating profitability and investment to illustrate how firm characteristics are measured. The example follows their inputs, then examines coverage and distributions. Portfolio formation and factor returns follow in 02–03.

### 2.1 Data and measurement conventions

The inputs are the linked monthly panel and independent annual accounts prepared in [00](00_crsp_compustat_data_preparation.ipynb). `permno` identifies a security and `gvkey` a company. Accounting values are in millions of dollars, prices in dollars and shares in thousands. Month-start dates label monthly observations; they do not indicate when returns became known.

I use June 2020 securities on NYSE, AMEX and Nasdaq with positive market equity, without filtering on realized returns. Fiscal-2019 accounts are assigned to holdings starting in July 2020 under the stated availability convention. Investment also requires fiscal-2018 accounts. Historical publication dates are unavailable, and the linked universe can differ from official French eligibility.

| Characteristic | Measurement | Inputs for June 2020 |
|---|---|---|
| Size | Market equity | June 2020 price and shares |
| Book-to-market | Book equity / market equity | Fiscal-2019 accounts and December 2019 market equity |
| Operating profitability | Operating surplus / book equity | Fiscal-2019 accounts |
| Investment | Annual asset growth | Fiscal-2018 and fiscal-2019 accounts |

**Size and book-to-market.** Market equity is $ME_{i,t}=|PRC_{i,t}|SHROUT_{i,t}$. The absolute price removes CRSP's quote-sign convention. Dividing the product by 1,000 gives millions, matching the accounting inputs. Size uses June market equity; BM uses the previous December denominator:

$$BE=SEQ+TXDITC-PS,\qquad BM_{i,2020}=\frac{BE_{i,2019}}{ME^{Dec}_{i,2019}}.$$

$SEQ$ is shareholders' equity, $TXDITC$ deferred taxes and investment tax credit, and $PS$ preferred stock. Preferred stock uses redemption value first, liquidation value second and carrying value last. Missing deferred taxes and preferred stock become zero; nonpositive book equity becomes missing. Higher BM means more book equity per dollar of market value, without establishing undervaluation.

**Profitability and investment.** The measurements are

$$OP_{i,2020}=\frac{REVT_{i,2019}-COGS_{i,2019}-XSGA_{i,2019}-XINT_{i,2019}}{BE_{i,2019}},\qquad
INV_{i,2020}=\frac{AT_{i,2019}-AT_{i,2018}}{AT_{i,2018}}.$$

OP divides revenue less cost of goods sold, SG&A and interest by book equity. It requires revenue, COGS and positive book equity, while missing SG&A and interest become zero. This expense rule differs from French's published convention. Investment measures asset growth, requires consecutive fiscal years and positive lagged assets, and equals 10% growth when the ratio is 0.10.

In [ ]:
FORMATION_YEAR = 2020
formation_month = pd.Timestamp(FORMATION_YEAR, 6, 1)
formation_date = pd.Timestamp(FORMATION_YEAR, 6, 30)
december_month = pd.Timestamp(FORMATION_YEAR-1, 12, 1)
raw, original_annual = load_prepared_data(PROCESSED, RAW_WRDS,
    start=f"{FORMATION_YEAR-1}-01-01", end=formation_month)
assert not raw.duplicated(["permno", "mdate"]).any()
# Independent annual inputs, not accounts recovered from a future July row.
annual = original_annual.loc[
    original_annual.fyear.isin([FORMATION_YEAR-2, FORMATION_YEAR-1]) &
    original_annual.available_from.le(pd.Timestamp(FORMATION_YEAR, 7, 1))
].copy()
assert not annual.duplicated(["gvkey", "fyear"]).any()
api_panel = raw.rename(columns={"date": "crsp_date", "mdate": "date"})
measured = fama_french_characteristics(api_panel, annual_data=annual)
formation = measured.loc[measured.date.eq(formation_month) & measured.exchcd.isin([1,2,3]) &
                          measured.me.gt(0), ['permno','gvkey','me','me_dec','bm','op','inv']].copy()
formation['size_millions'] = formation.me/1000
formation = formation.sort_values('permno').reset_index(drop=True)
assert len(formation)>0 and formation.permno.is_unique
measure_columns = ['size_millions','bm','op','inv']
print(f'June {FORMATION_YEAR}: {len(formation):,} eligible security records in the locally prepared linked universe.')
print('Independent annual records use the assumed July holding-availability convention, not verified filing dates.')


### 2.2 Computing characteristics from observed records

I use `fama_french_characteristics` to align the dated accounting and market inputs and calculate the measurements. Cleaning and historical linking follow 00; the annual inputs remain subject to the availability convention above.

The tables show the first three distinct companies ordered by security identifier among June records with all four measurements. This gives a reproducible illustration without selecting on future returns or attractive characteristic values. The first panel shows accounting inputs; the second shows dated market inputs and calculated outputs. Company names are unavailable in the local annual export.

In [ ]:
complete = formation.loc[formation[measure_columns].notna().all(axis=1)]
examples = complete.drop_duplicates('gvkey').head(3).copy()
assert len(examples)==3
accounts_current = annual.loc[annual.fyear.eq(FORMATION_YEAR-1)].set_index('gvkey')
accounts_prior = annual.loc[annual.fyear.eq(FORMATION_YEAR-2)].set_index('gvkey')
worked = examples.merge(accounts_current.reset_index(),on='gvkey',how='left',validate='many_to_one')
worked = worked.merge(accounts_prior[['at','datadate']].rename(columns={'at':'prior_at',
    'datadate':'prior_datadate'}).reset_index(),on='gvkey',how='left',validate='many_to_one')
market_inputs = raw.loc[raw.mdate.eq(december_month),['permno','prc','shrout']].rename(
    columns={'prc':'dec_price','shrout':'dec_shares'})
market_inputs = market_inputs.merge(raw.loc[raw.mdate.eq(formation_month),['permno','prc','shrout']]
    .rename(columns={'prc':'june_price','shrout':'june_shares'}),on='permno',validate='one_to_one')
worked = worked.merge(market_inputs,on='permno',validate='one_to_one')
worked['preferred'] = worked.pstkrv.combine_first(worked.pstkl).combine_first(worked.pstk).fillna(0)
worked['book_equity'] = worked.seq + worked.txditc.fillna(0) - worked.preferred
worked['surplus'] = worked.revt-worked.cogs-worked.xsga.fillna(0)-worked.xint.fillna(0)
print('Observed accounting inputs: dollar amounts in millions. Missing expenses remain visible.')
account_panel = worked[['permno','gvkey','fyear','datadate','seq','txditc','preferred','revt','cogs',
                        'xsga','xint','prior_datadate','prior_at','at']].copy()
# Transposing keeps three records readable while exposing every input.
display(account_panel.set_index('permno').T)
print('Dated market inputs and measured characteristics: market equity in $m, other fields are ratios.')
result_panel = worked[['permno','gvkey','dec_price','dec_shares','june_price','june_shares',
                        'me_dec','size_millions','book_equity','bm','op','inv']].rename(
    columns={'dec_price':'December price ($)', 'dec_shares':'December shares (000s)',
             'june_price':'June price ($)', 'june_shares':'June shares (000s)',
             'me_dec':'December ME ($m)','size_millions':'June ME ($m)','book_equity':'BE ($m)',
             'bm':'BM','op':'OP','inv':'INV'})
display(result_panel.set_index('permno').round(6).T)
r = worked.iloc[0]
print(f"Security {int(r.permno)}, company {r.gvkey}: BE = {r.seq:.3f} + {r.txditc if pd.notna(r.txditc) else 0:.3f} "
      f"− {r.preferred:.3f} = {r.book_equity:.3f} million.")
print(f"BM = {r.book_equity:.3f}/{r.me_dec:.3f} = {r.bm:.6f}.")
print(f"OP = {r.surplus:.3f}/{r.book_equity:.3f} = {r.op:.6f}.")
print(f"INV = ({r['at']:.3f} − {r.prior_at:.3f})/{r.prior_at:.3f} = {r.inv:.6f} ({100*r.inv:.3f}%).")


The records illustrate how the same rules produce different rankings across size, valuation, profitability and investment. A firm with greater market value need not have higher profitability or asset growth. Missing expenses follow the stated calculation rule rather than representing observed zeros.

The dates and denominators matter. June market value determines size, while December market value enters BM. OP measures profitability relative to equity, so it should not be read as a sales margin. These differences describe firms without establishing their expected returns.

In [ ]:
# Independent formulas and dated joins verify every eligible security, not just the examples.
manual_accounts = annual.copy()
preferred_manual = np.select([manual_accounts.pstkrv.notna(), manual_accounts.pstkl.notna()],
    [manual_accounts.pstkrv.to_numpy(dtype=float,na_value=np.nan),
     manual_accounts.pstkl.to_numpy(dtype=float,na_value=np.nan)],
    default=manual_accounts.pstk.fillna(0).to_numpy(dtype=float))
manual_accounts['BE_check'] = (manual_accounts.seq + manual_accounts.txditc.fillna(0)-preferred_manual)
manual_accounts['BE_check'] = manual_accounts.BE_check.where(manual_accounts.BE_check.gt(0))
manual_current = manual_accounts.loc[manual_accounts.fyear.eq(FORMATION_YEAR-1)].copy()
manual_current = manual_current.merge(accounts_prior[['at']].rename(columns={'at':'lag_assets'}).reset_index(),
                                    on='gvkey',how='left',validate='one_to_one')
manual_current['OP_check'] = ((manual_current.revt-manual_current.cogs-manual_current.xsga.fillna(0)-
    manual_current.xint.fillna(0))/manual_current.BE_check).where(
    manual_current.revt.notna() & manual_current.cogs.notna() & manual_current.BE_check.gt(0))
manual_current['INV_check'] = ((manual_current['at']-manual_current.lag_assets)/manual_current.lag_assets).where(
    manual_current.lag_assets.gt(0))
dec_raw = raw.loc[raw.mdate.eq(december_month),['permno','prc','shrout']].copy()
dec_raw['DEC_check'] = (dec_raw.prc.abs()*dec_raw.shrout/1000).where(dec_raw.shrout.gt(0))
check = formation.merge(manual_current[['gvkey','BE_check','OP_check','INV_check']],on='gvkey',
                        how='left',validate='many_to_one').merge(dec_raw[['permno','DEC_check']],
                        on='permno',how='left',validate='one_to_one')
check['BM_check'] = (check.BE_check/check.DEC_check).where(check.DEC_check.gt(0))
for api_name, independent_name in [('bm','BM_check'),('op','OP_check'),('inv','INV_check')]:
    np.testing.assert_allclose(check[api_name].to_numpy(dtype=float,na_value=np.nan),
        check[independent_name].to_numpy(dtype=float,na_value=np.nan),rtol=1e-12,atol=1e-12,equal_nan=True)
june_raw = raw.loc[raw.mdate.eq(formation_month)].set_index('permno').loc[formation.permno]
np.testing.assert_allclose(formation.size_millions.to_numpy(dtype=float),
                          (june_raw.prc.abs()*june_raw.shrout/1000).to_numpy(dtype=float))
for name in measure_columns:
    assert np.isfinite(formation[name].dropna().to_numpy(dtype=float)).all()
print(f'Independent dated-input arithmetic agrees for all {len(formation):,} eligible security records.')


### 2.3 Measurement coverage

A usable market observation does not guarantee usable accounting characteristics. Each measurement requires different inputs, so I report its available observations against the same June-universe denominator.

The table shows counts and coverage shares, followed by medians and interquartile ranges. These summarize typical values and dispersion without letting extreme ratios dominate an average. They are not portfolio breakpoints. Each characteristic retains its own available sample.

In [ ]:
labels = {'size_millions':'Size ($m)','bm':'Book-to-market (ratio)',
          'op':'Operating profitability (ratio)','inv':'Investment (ratio)'}
coverage_rows = []
for name in measure_columns:
    v = formation[name].dropna().astype(float)
    coverage_rows.append({'Characteristic':labels[name], 'Usable records':len(v),
        'Coverage (%)':100*len(v)/len(formation), '25th percentile':v.quantile(.25),
        'Median':v.median(), '75th percentile':v.quantile(.75)})
coverage = pd.DataFrame(coverage_rows).set_index('Characteristic')
display(coverage.round(4))
for name, row in zip(measure_columns,coverage.itertuples(index=False)):
    v = formation[name].dropna().to_numpy(dtype=float)
    assert row[0] == np.isfinite(v).sum()
    np.testing.assert_allclose([row[2],row[3],row[4]],np.percentile(v,[25,50,75]))
print(f'{len(complete):,} securities have all four measurements. That subset selects the worked records only.')


### 2.4 Cross-sectional distributions

The distributions show how firms differ before we form portfolios. Size uses a logarithmic horizontal scale, while BM, OP and investment retain their ratio units.

For readability, the latter panels show their 1st–99th percentile ranges and report how many observations fall outside them. This changes only the display; the calculations and coverage table retain the full observations.

In [ ]:
fig, axes = plt.subplots(2,2,figsize=(11,7.5),layout='constrained')
display_limits = {}
for ax, name, color in zip(axes.flat,measure_columns,[NAVY,BLUE,TEAL,AMBER]):
    v = formation[name].dropna().astype(float)
    if name=='size_millions':
        bins = np.geomspace(v.min(),v.max(),31)
        ax.hist(v,bins=bins,color=color,edgecolor='white',linewidth=.35)
        ax.set_xscale('log')
        outside = 0
    else:
        lo, hi = v.quantile([.01,.99])
        display_limits[name] = (float(lo),float(hi))
        shown = v.loc[v.between(lo,hi)]
        outside = len(v)-len(shown)
        ax.hist(shown,bins=30,range=(lo,hi),color=color,edgecolor='white',linewidth=.35)
        if lo<=0<=hi:
            ax.axvline(0,color=GRAY,lw=.9,ls='--')
    ax.set(xlabel=labels[name],ylabel='Number of security records',
           title=f'{labels[name].split(" (")[0]}: n={len(v):,}')
    ax.text(.98,.96,f'{outside} outside displayed range',transform=ax.transAxes,
            ha='right',va='top',fontsize=9,
            bbox={'facecolor':'white','edgecolor':'none','alpha':.85})
    ax.grid(axis='y',alpha=.15)
fig.suptitle(f'Prepared linked universe at June {FORMATION_YEAR}: characteristic distributions',fontsize=13)
plt.show()
for name,(lo,hi) in display_limits.items():
    v=formation[name].dropna().astype(float)
    print(f'{labels[name]}: full observed range [{v.min():.4g}, {v.max():.4g}], '
          f'displayed [{lo:.4g}, {hi:.4g}].')


The vertical axes count securities rather than capital invested. Median security market equity is about USD 632 million, with an interquartile range from USD 136 million to USD 3.22 billion. This size range makes the choice between equal and value weights consequential.

Median BM is approximately 0.505, OP is 0.152 and annual investment is 7.12%. The 25th percentiles of OP and investment are negative, showing that operating losses and asset contraction are not confined to extreme observations. The panels have different sample sizes because each measurement requires different inputs. These distributions provide the comparisons for later sorts, rather than forecasts of returns.

The example connects the inputs to the measurements and shows which firms we can compare. [02](02_portfolio_formation.ipynb) takes the next step: defining portfolio assignments and weights from these characteristics.

## References

- Fama, E. F., and French, K. R. (2015). A five-factor asset pricing model. *Journal of Financial Economics*, 116(1), 1–22. [DOI](https://doi.org/10.1016/j.jfineco.2014.10.010).
- Kenneth French, [five-factor construction documentation](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html).
